# 🚀 RGB Evaluation Suite on Google Colab

This notebook guide helps you benchmark large language models (like **Llama 3 (8B)** and **Qwen 2.5 (7B/8B)**) on the **RGB (Retrieval-Augmented Generation Benchmark)** English datasets using your Google Colab VM (with GPU acceleration).

### ⚠️ Setup GPU Runtime First
Before running any cells, ensure you are connected to a GPU instance:
1. In the top menu, go to **Runtime** > **Change runtime type**.
2. Under *Hardware accelerator*, select **T4 GPU** (or any other available GPU).
3. Click **Save**.

## 📂 Step 1: Set Up Project Workspace

Choose **Option A** if you uploaded/cloned the workspace zip to Google Drive, or **Option B** if you want to upload the zipped folder directly to the Colab local file system.

In [ ]:
# === OPTION A: Run from Google Drive ===
# Mount Google Drive and navigate to the project directory
from google.colab import drive
drive.mount('/content/drive')

# Update this path to point to your ragstack directory in Drive
%cd /content/drive/MyDrive/ragstack

# If you uploaded 'ragstack_colab.zip' inside the 'ragstack' folder in Google Drive,
# run this command to unzip and extract all updated files directly into the current folder:
!unzip -o ragstack_colab.zip -d .

In [ ]:
# === OPTION B: Run from local Colab VM ===
# Upload your 'ragstack_colab.zip' archive using the file explorer sidebar on the left.
# Then run this cell to unzip and navigate to the workspace.
import os
if os.path.exists('/content/ragstack_colab.zip'):
    !unzip -o /content/ragstack_colab.zip -d /content/ragstack
    %cd /content/ragstack
else:
    print("Could not find ragstack_colab.zip. Please upload it first or use Option A above.")

## 📦 Step 2: Install Dependencies

Install the required packages. Colab includes standard PyTorch and Pandas libraries, but we need text generators and evaluation utility libraries.

In [ ]:
!pip install -r requirements.txt

## 🦙 Step 3: Deploy Ollama & Pull 8B Models

To evaluate **Llama 3 8B** and **Qwen 2.5 7B/8B** locally on the Colab GPU VM, we will download and run the Ollama background daemon, sleep to allow it to initialize, and pull the required model files.

In [ ]:
# 1. Install zstd (required by Ollama installer) and download/install Ollama
!apt-get update && apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama background process
import subprocess
import time
print("Starting Ollama server daemon...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5) # Wait for Ollama server to spin up

# 3. Pull the models of your choice
print("Pulling llama3:8b...")
!ollama pull llama3:8b

print("Pulling qwen2.5:7b...")
!ollama pull qwen2.5:7b

## ⚙️ Step 4: Run RGB Evaluations

Use the interactive configuration form below to specify evaluation parameters. Run the cell to execute the benchmark script.

In [ ]:
#@title RGB Benchmarking Configuration Form { run: "auto" }

#@markdown ### 📋 Dataset & Limits
dataset = "en" #@param ["en", "en_refine", "en_int", "en_fact"]
n_records = 10 #@param {type:"integer"} # Set to 0 or a very large number for complete dataset

#@markdown ### 🤖 Model under evaluation
generator = "llama3:8b" #@param ["llama3:8b", "qwen2.5:7b", "hf-small", "hf-large", "mock"]

#@markdown ### 🔧 Document Injection / Noise Parameters
passage_num = 5 #@param {type:"integer"}
noise_rate = 0.4 #@param {type:"number"} # Ratio of negative passages. Use 1.0 to test Negative Rejection.
correct_rate = 0.0 #@param {type:"number"} # Only used in 'en_fact' for counterfactual robustness.

#@markdown ### ⚖️ LLM-as-a-judge (For Rejection & Factual Detection rates)
use_llm_judge = True #@param {type:"boolean"}
judge_generator = "llama3:8b" #@param ["llama3:8b", "qwen2.5:7b"]

n_records_flag = f"--n_records {n_records}" if n_records > 0 else ""
judge_flag = "--use_llm_judge" if use_llm_judge else ""
judge_gen_flag = f"--judge_generator {judge_generator}" if use_llm_judge and judge_generator else ""

!python scripts/run_rgb_eval.py \
  --dataset {dataset} \
  --generator {generator} \
  --passage_num {passage_num} \
  --noise_rate {noise_rate} \
  --correct_rate {correct_rate} \
  {n_records_flag} \
  {judge_flag} \
  {judge_gen_flag}

## 📊 Step 5: Read Saved Results

Let's load and display the exact summary metrics of the run configured above.

In [ ]:
import json
import os
import glob
from datetime import datetime

# Reconstruct the expected filename based on Step 4's parameters
output_filename = f"prediction_{dataset}_{generator}_noise{noise_rate}_passage{passage_num}_correct{correct_rate}"
if use_llm_judge:
    output_filename += "_judge"
    
summary_path = os.path.join("eval/results/rgb", f"{output_filename}_summary.json")

if os.path.exists(summary_path):
    print(f"Displaying summary for run: {summary_path}\n")
    with open(summary_path, 'r', encoding='utf-8') as f:
        summary_data = json.load(f)
    print(json.dumps(summary_data, indent=4))
    
    # Generate and write individual summary markdown for committing in the repo
    md_lines = [
        f"# 🏆 RGB Evaluation Summary - {output_filename}",
        "",
        "This is an automatically generated summary of the RGB evaluation run.",
        "",
        "### ⚙️ Run Configuration",
        f"- **Dataset:** `{summary_data.get('dataset')}`",
        f"- **Generator:** `{summary_data.get('generator')}`",
        f"- **Passage Num:** {summary_data.get('passage_num')}",
        f"- **Noise Rate:** {summary_data.get('noise_rate')}",
        f"- **Correct Rate:** {summary_data.get('correct_rate')}",
        f"- **Total Records:** {summary_data.get('total_records')}",
        "",
        "### 📊 Metrics",
        f"| Metric | Value |",
        f"| --- | --- |",
        f"| **Accuracy (Acc)** | {summary_data.get('accuracy', 0.0):.2%} |",
        f"| **Rejection Rate (Rej)** | {summary_data.get('rejection_rate', 0.0):.2%} |",
    ]
    
    if '_fact' in dataset:
        md_lines.extend([
            f"| **Error Detection (ED)** | {summary_data.get('error_detection_rate', 0.0):.2%} |",
            f"| **Error Correction (CR)** | {summary_data.get('error_correction_rate', 0.0):.2%} |",
        ])
        
    md_lines.append("")
    md_lines.append(f"*Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*")
    
    ind_md_path = os.path.join("eval/results/rgb", f"{output_filename}_summary.md")
    with open(ind_md_path, 'w', encoding='utf-8') as f_ind:
        f_ind.write("\n".join(md_lines))
    print(f"\nGenerated individual markdown summary: {ind_md_path}")
    
    # Update/Generate cumulative RGB evaluation leaderboard
    rgb_dir = "eval/results/rgb"
    summary_files = glob.glob(os.path.join(rgb_dir, "*_summary.json"))
    
    leaderboard_data = []
    for filepath in summary_files:
        try:
            with open(filepath, 'r', encoding='utf-8') as f_json:
                data = json.load(f_json)
                mtime = os.path.getmtime(filepath)
                dt = datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S')
                data['timestamp'] = dt
                data['run_id'] = os.path.basename(filepath).replace("_summary.json", "")
                leaderboard_data.append(data)
        except Exception as e:
            print(f"Warning: Failed to load {filepath}: {e}")
            
    if leaderboard_data:
        leaderboard_data.sort(key=lambda x: (x.get('dataset', ''), -x.get('accuracy', 0.0)))
        
        lead_lines = [
            "# 🏆 RGB Evaluation Leaderboard",
            "",
            "This is an automatically generated leaderboard of the RGB evaluation runs, grouped by dataset.",
            "This file is tracked under Git version control to keep a record of your experimental history.",
            "",
            "| Run ID | Dataset | Generator | Passage Num | Noise Rate | Correct Rate | Accuracy | Rejection Rate | Error Detection | Error Correction | Last Run |",
            "| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |"
        ]
        
        for item in leaderboard_data:
            run_id = item.get('run_id', '')
            ds = item.get('dataset', '')
            gen = item.get('generator', '')
            p_num = item.get('passage_num', 0)
            n_rate = item.get('noise_rate', 0.0)
            c_rate = item.get('correct_rate', 0.0)
            acc = item.get('accuracy', 0.0)
            rej = item.get('rejection_rate', 0.0)
            ed = item.get('error_detection_rate', 0.0)
            cr = item.get('error_correction_rate', 0.0)
            ts = item.get('timestamp', '')
            
            ed_str = f"{ed:.2%}" if '_fact' in ds else "N/A"
            cr_str = f"{cr:.2%}" if '_fact' in ds else "N/A"
            
            lead_lines.append(
                f"| `{run_id}` | {ds} | {gen} | {p_num} | {n_rate} | {c_rate} | **{acc:.2%}** | {rej:.2%} | {ed_str} | {cr_str} | {ts} |"
            )
            
        lead_lines.append("")
        lead_lines.append(f"*Last updated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}*")
        
        lead_md_path = os.path.join("eval", "evaluation_summary_rgb.md")
        with open(lead_md_path, 'w', encoding='utf-8') as f_lead:
            f_lead.write("\n".join(lead_lines))
        print(f"Updated cumulative RGB leaderboard: {lead_md_path}")
else:
    print(f"Could not find summary file at: {summary_path}")
    print("Please verify that the configuration matches the completed run in Step 4.")